# PyPSA's optimisation model, written as math (PR #2128)

> **Throwaway notebook for reviewers.** It is not part of the release or the docs.

**In one sentence:** `n.optimize()` now builds its model from math written in YAML files, not from Python code.

| Question | Before | After | Section |
|---|---|---|---|
| Where is the math? | In Python, in the `define_*` functions | In YAML files, one entry per variable, constraint or cost term | 1.1 |
| What changes for user code? | | The call and the results stay. The layout of `n.model` changes. | 1.2 |
| How do I read the math? | Read the linopy code | Print it as equations | 1.3 |
| How do I change the math? | Write linopy code in `extra_functionality` | Pass a patch: `n.optimize(overrides=[...])` | 1.4 |
| How do I keep a change? | Run the code again at each build | Store the patch on the network. A saved file keeps it. | 1.5 |
| How do I add a technology of several components? | Add the components in a helper and their math in `extra_functionality` | Define a *composite* type that holds both | 1.6 |
| Which input does the math refuse? | Checks in Python, apart from the math | `assumptions:` in the YAML, next to the math | 1.7 |

The notebook has two parts:

1. **What a user gets.** Each feature on a small network. This part uses the public API only.
2. **How it works inside.** The modules, the data flow and the technical decisions that the YAML does not show. This part uses internal names, which can change.

Every output comes from a run on the PR branch.

The first cell silences the logs, before the import of PyPSA.
The second cell loads the libraries and defines three helpers: `show` prints entries as YAML, `read` loads one YAML file of the spec, and `show_error` runs a call that must fail and prints the error.
Two `future` options stop the warnings about the defaults of PyPSA 2.0.

In [1]:
import logging

logging.getLogger("pypsa").setLevel(logging.ERROR)
logging.getLogger("linopy").setLevel(logging.ERROR)

In [2]:
import inspect
import json
import tempfile
import warnings
from collections.abc import Callable, Mapping
from pathlib import Path
from typing import Any

import linopy
import mathspec
import pandas as pd
import yaml
from IPython.display import Math, display

import pypsa
from pypsa.optimization import Override

pypsa.options.future.string_dtype = False
pypsa.options.future.exclude_objective_constant = True
warnings.filterwarnings("ignore", category=linopy.EvolvingAPIWarning)

SPEC_DIR = Path(pypsa.__file__).parent / "optimization" / "spec"


def show(section: Mapping[str, Any], *keys: str) -> None:
    entries = {key: section[key] for key in keys}
    print(yaml.safe_dump(entries, sort_keys=False, width=100))


def read(path: str) -> dict[str, Any]:
    return yaml.safe_load((SPEC_DIR / path).read_text())


def show_error(action: Callable[[], object]) -> None:
    try:
        action()
    except Exception as error:
        print(f"{type(error).__name__}: {error}")
    else:
        raise AssertionError("the call did not fail")


print("linopy", linopy.__version__, "| mathspec", mathspec.__version__)

linopy 0.10.0.post1.dev122+g4e78d4ea5 | mathspec 0.4.1.dev38+gcc5311511


# Part 1: What a user gets

## 1.1 The idea: the math is data

```
  YAML files (the math)          pypsa.Network (the data)
          │                               │
          ▼                               ▼
    one spec object            arrays that the spec asks for
          │                               │
          └───────────► linopy ◄──────────┘
                            │
                            ▼
                      linopy.Model
```

The **spec** (specification) is one object that holds every entry of the math.
The library `mathspec` defines its format.
linopy builds a model from a spec and the data.

Three rules hold the design together:

| Rule | What it means |
|---|---|
| The math is in YAML | Each variable, constraint and cost term is an entry in a file under `pypsa/optimization/spec/`. |
| Python prepares the data | Python turns network data into the arrays that the entries name. |
| linopy builds the model | `linopy.Model.from_spec` reads the spec and the arrays. PyPSA then sets the layout and the scaling of the model. |

The rest of Part 1 follows from the first rule: you can read data, print it, patch it, store it and reuse it.

Two limits:

- **Not all math is in YAML.** Python still adds the variable `objective_constant` (`set_objective_constant` in `optimize.py`) and the deprecated `Bus` limits `nom_min_<carrier>` and `nom_max_<carrier>` (`define_nominal_constraints_per_bus_carrier` in `global_constraints.py`).
- **The PR needs versions of linopy and mathspec that are not released.** `pyproject.toml` pins both to a git commit. The cell above prints the versions of this run.

## 1.2 What stays and what changes

The example is a network with two buses: wind in the north, and gas, a store and the demand in the south.
The model chooses the capacity of wind, gas and the store.

In [3]:
def toy() -> pypsa.Network:
    n = pypsa.Network()
    n.set_snapshots(pd.date_range("2030-01-01", periods=4, freq="h"))
    n.add("Carrier", ["wind", "gas", "AC"], co2_emissions=[0, 0.4, 0])
    n.add("Bus", ["north", "south"], carrier="AC")
    line = {"bus0": "north", "bus1": "south", "x": 0.1, "r": 0.01, "s_nom": 50}
    n.add("Line", "north-south", carrier="AC", **line)
    n.add("Load", "demand", bus="south", p_set=[60, 80, 100, 70])
    wind = {"carrier": "wind", "capital_cost": 80, "p_max_pu": [0.9, 0.5, 0.2, 0.7]}
    gas = {"carrier": "gas", "capital_cost": 40, "marginal_cost": 60}
    n.add("Generator", "wind", bus="north", p_nom_extendable=True, **wind)
    n.add("Generator", "gas", bus="south", p_nom_extendable=True, **gas)
    store = {"e_nom_extendable": True, "capital_cost": 10, "e_cyclic": True}
    n.add("Store", "storage", bus="south", **store)
    return n


n = toy()
n.optimize(log_to_console=False)
print(f"objective: {n.objective:,.2f}")
n.generators.p_nom_opt.round(1)

objective: 17,414.29


name
wind    71.4
gas     40.0
Name: p_nom_opt, dtype: float64

**What stays:** the call, the results on the network, and most names of the variables and constraints in `n.model`. The release notes list the names that change.
A test compares the optimum of 24 networks with the values from `master` (2.11).

**What changes:** the layout of `n.model`. Code that reads or extends the model can need a change. Two examples:

In [4]:
m = n.model
dispatch = m.variables["Generator-p"]
scenarios = dispatch.coords["scenario"].values.tolist()
print("Generator-ext-p-upper" in m.constraints)
print("Generator-p:    ", dispatch.dims, "with the scenarios", scenarios)
print("Generator-p_nom:", m.variables["Generator-p_nom"].dims)

True
Generator-p:     ('name', 'scenario', 'snapshot') with the scenarios ['base']
Generator-p_nom: ('name',)


- The network has no scenarios, but `Generator-p` has a `scenario` dimension with the one label `"base"`. Before, it had none.
- `name` is the first dimension. Before, `snapshot` came first.

The release notes list all changes of the layout. Section 2.9 shows where they come from.

**What is gone:** the `define_*` functions. The modules `pypsa.optimization.constraints` and `pypsa.optimization.variables` are stubs now.
An import of a removed name fails, and the error names the entries that replace it:

In [5]:
def old_import() -> object:
    from pypsa.optimization.constraints import define_nodal_balance_constraints

    return define_nodal_balance_constraints


show_error(old_import)

ImportError: `pypsa.optimization.constraints.define_nodal_balance_constraints` was removed in PyPSA 1.5.0. `n.optimize()` now builds the model from the specification in `pypsa/optimization/spec/`. Its constraints are `Bus_nodal_balance` (system.yaml). To change the model, pass `overrides` to `n.optimize()`, see https://docs.pypsa.org/latest/user-guide/optimization/custom-constraints/#overriding-the-formulation


## 1.3 Read the math

`n.optimize.spec()` returns the spec that `n.optimize()` builds from.
It is plain data, so you can count its entries:

In [6]:
s = n.optimize.spec()
kinds = (
    "dimensions",
    "relations",
    "parameters",
    "conditions",
    "variables",
    "expressions",
    "constraints",
    "piecewise",
    "assumptions",
)
print(len(n.model.constraints), "constraints in the model of the toy network")
pd.Series({kind: len(getattr(s, kind)) for kind in kinds}, name="entries in the spec")

12 constraints in the model of the toy network


dimensions      20
relations       28
parameters     326
conditions      88
variables       70
expressions    158
constraints    235
piecewise       27
assumptions     51
Name: entries in the spec, dtype: int64

| Kind | An entry is |
|---|---|
| dimension | an index set, as `snapshot` or `Generator` (the units of a component) |
| relation | a map from one dimension into another, as `Generator_bus` from each generator to its bus |
| parameter | input data, as `Generator_p_max_pu` |
| condition | a named yes/no mask, as `Generator_ext` (the extendable active units) |
| variable, constraint | as in every optimisation model |
| expression | a named term that other entries reuse, as `total_cost` |
| piecewise | a piecewise linear curve, as a `marginal_cost` curve |
| assumption | a rule that the input data must obey (1.7) |

The spec holds the entries of every component and feature. The model of the toy network holds only the 12 constraints that have a row there.

One entry, as the spec holds it:

In [7]:
show(s.to_dict()["constraints"], "Generator_ext_p_nom_upper", "Generator_ext_p_upper")

Generator_ext_p_nom_upper:
  dims:
  - Generator
  where: Generator_p_nom_extendable AND Generator_active AND Generator_p_nom_max
  expression: Generator_p_nom_ext <= Generator_p_nom_max
  description: the chosen build is at most the cap; an infinite cap means no row
Generator_ext_p_upper:
  dims:
  - Generator
  - scenario
  - snapshot
  where: Generator_ext AND NOT Generator_committable
  expression: Generator_p <= Generator_p_max_pu * (Generator_p_nom_ext - Generator_maintenance_pu * Generator_maintenance_capacity)
  description: an extendable unit that is not committable runs at most at its maximum of the chosen build



`dims` lists the sets that the constraint runs over. `where` says which members get a row.

`mathspec` prints the same entries as equations. The file `symbols.yaml` gives each name a short symbol:

In [8]:
notation = mathspec.SymbolTable.load(read("symbols.yaml"))
for key in ("Generator_ext_p_nom_upper", "Generator_ext_p_upper"):
    display(Math(mathspec.typeset_line(s, key, "latex", symbols=notation)))
document = mathspec.typeset(s, "latex", symbols=notation, standalone=True)
print(f"the whole spec as one LaTeX document: {len(document.splitlines())} lines")

<IPython.core.display.Math object>

<IPython.core.display.Math object>

the whole spec as one LaTeX document: 1446 lines


The equations and the model come from one object, `s`. No second copy of the math exists that can go out of date.

## 1.4 Change the math for one call

An **override** is a patch for the spec. It is a dict, YAML text or the path of a YAML file.
`n.optimize(overrides=[...])` lays the patches over the spec for this one build.
A patch can change fields of an entry, add an entry, or remove an entry with `null`.
The linopy code in `extra_functionality` still runs after the build (not shown here).

**Example 1: change one field.** Extendable generators can use only a share of their capacity.
The patch states only the `expression`. The entry keeps its other fields:

In [9]:
def capped(share: float) -> Override:
    expression = f"Generator_p <= {share} * Generator_p_max_pu * Generator_p_nom_ext"
    return {"constraints": {"Generator_ext_p_upper": {"expression": expression}}}


HALF = capped(0.5)
n = toy()
n.optimize(overrides=[HALF], log_to_console=False)
print(f"objective: {n.objective:,.2f}\n")
patched = n.optimize.spec(overrides=[HALF])
show(patched.to_dict()["constraints"], "Generator_ext_p_upper")

objective: 23,966.67

Generator_ext_p_upper:
  dims:
  - Generator
  - scenario
  - snapshot
  where: Generator_ext AND NOT Generator_committable
  expression: Generator_p <= 0.5 * Generator_p_max_pu * Generator_p_nom_ext
  description: an extendable unit that is not committable runs at most at its maximum of the chosen build



The objective rises from 17,414.29 (1.2) to 23,966.67. `pypsa.optimization.Override` is the type for functions that return an override.

**Example 2: add a parameter and a constraint.** Wind must run for a minimum of full-load hours.

The new parameter `Generator_full_load_hours_min` has the form `<Component>_<attribute>`.
So PyPSA reads its values from the column `full_load_hours_min` of the generators:

In [10]:
FULL_LOAD_HOURS = """
parameters:
  Generator_full_load_hours_min:
    dims: [Generator]
constraints:
  Generator_full_load_hours:
    dims: [scenario, Generator]
    where: Generator_full_load_hours_min > 0 AND Generator_p_nom_extendable
    expression: >-
      sum(Generator_p * snapshot_weighting_generators, over=snapshot)
      >= Generator_full_load_hours_min * Generator_p_nom_ext
"""

rows = {}
for hours in (0.0, 2.0, 2.2):
    n = toy()
    n.c.generators.static["full_load_hours_min"] = [hours, 0.0]
    n.optimize(overrides=[FULL_LOAD_HOURS], log_to_console=False)
    capacity = n.generators.p_nom_opt
    rows[hours] = {
        "objective": n.objective,
        "wind capacity": capacity["wind"],
        "gas capacity": capacity["gas"],
        "wind full-load hours": n.generators_t.p["wind"].sum() / capacity["wind"],
    }
pd.DataFrame(rows).T.rename_axis("minimum for wind").round(1)

,objective,wind capacity,gas capacity,wind full-load hours
minimum for wind,,,,
0.0,17414.3,71.4,40.0,2.1
2.0,17414.3,71.4,40.0,2.1
2.2,17575.0,62.5,43.1,2.2


Without a minimum, wind runs 2.1 full-load hours. So a minimum of 2.0 changes nothing.
A minimum of 2.2 binds: the model builds less wind and more gas, and the cost rises.

**Which keys can an override rely on?** The docs promise *stable keys* (`docs/user-guide/optimization/custom-constraints.md`):

- Stable are the keys of variables, constraints and dimensions, the objective, and the parameters that carry the name of an attribute.
- All other keys are helpers and can change in any release. `snapshot_weighting_generators` in example 2 is such a helper.
- Until the first release with overrides, every key can still change.

## 1.5 Keep a change with the network

An override in `n.optimize(overrides=[...])` holds for one call.
`n.optimize.overrides.register` stores it on the network. Then every build of the network applies it.
A copy and a saved file keep it:

In [11]:
n = toy()
n.c.generators.static["full_load_hours_min"] = [2.2, 0.0]
n.optimize.overrides.register(FULL_LOAD_HOURS, name="full_load_hours")
copied = n.copy()
with tempfile.TemporaryDirectory() as folder:
    n.export_to_netcdf(Path(folder) / "toy.nc")
    loaded = pypsa.Network(Path(folder) / "toy.nc")

for label, network in (("original", n), ("copy", copied), ("loaded file", loaded)):
    network.optimize(log_to_console=False)
    names = list(network.optimize.overrides)
    print(f"{label:12} {names}  objective: {network.objective:,.2f}")

original     ['full_load_hours']  objective: 17,575.00


copy         ['full_load_hours']  objective: 17,575.00


loaded file  ['full_load_hours']  objective: 17,575.00


All three builds find 17,575.00. That is the optimum with the minimum of 2.2 hours (1.4).

`n.optimize.overrides` reads as a mapping from a name to a patch.
`register` checks the patch against the spec at once. So a wrong key fails here, and not at a later build.
It also refuses a patch that the network stores already:

In [12]:
TYPO = {"constraints": {"Generator_ext_p_uper": None}}
overrides = n.optimize.overrides
print(overrides, "with", list(overrides["full_load_hours"]["constraints"]))
show_error(lambda: overrides.register(TYPO, name="typo"))
show_error(lambda: overrides.register(FULL_LOAD_HOURS, name="again"))
overrides.unregister("full_load_hours")
print(overrides)

Overrides ['full_load_hours'] with ['Generator_full_load_hours']
LanguageError: the stored override 'typo' removes the constraint 'Generator_ext_p_uper', which its base does not declare. Delete the removal from the patch, or fix the name. Did you mean 'Generator_ext_p_upper'?
ValueError: The override 'again' is registered already as 'full_load_hours'. Use that name, or unregister it first.
Overrides []


**Order.** A build lays its patches over the spec in a fixed order. A later layer wins:

| Layer | Set by | Kept with the network? |
|---|---|---|
| 1. Linearised unit commitment | `n.optimize(linearized_unit_commitment=True)` | no |
| 2. Piecewise options | `n.optimize(piecewise_options=...)` | no |
| 3. Composite types (1.6) | `n.composites.register(...)` | yes |
| 4. Stored overrides, in the order of registration | `n.optimize.overrides.register(...)` | yes |
| 5. Per-call overrides, in the order of the list | `n.optimize(overrides=[...])` | no |

A stored cap of 0.5 and a per-call cap of 0.8 show the order:

In [13]:
n = toy()
n.optimize.overrides.register(capped(0.5), name="half")
specs = {
    "stored only": n.optimize.spec(),
    "stored and per-call": n.optimize.spec(overrides=[capped(0.8)]),
}
for label, spec_of_build in specs.items():
    expression = spec_of_build.constraints["Generator_ext_p_upper"].expression
    print(f"{label:20} {expression}")

stored only          Generator_p <= 0.5 * Generator_p_max_pu * Generator_p_nom_ext
stored and per-call  Generator_p <= 0.8 * Generator_p_max_pu * Generator_p_nom_ext


Two limits:

- A network keeps layers 3 and 4, but not the arguments of a call. Pass `linearized_unit_commitment` and `piecewise_options` again after you load a file.
- A stored override must fit the default spec. An override of an entry that only layer 1 adds cannot be stored; pass it per call (`test_an_override_of_linearised_unit_commitment_stays_per_call`).

## 1.6 Add a technology as one piece: composites

A **composite type** is a group of components and its math, written once. You add the group as often as you need.

A battery, for example, is a DC bus, a charger, a discharger and a store.
The charger and the discharger share one inverter, so they must have the same rating.
One definition holds the components as templates and the math as spec entries:

In [14]:
BATTERY = """
name: battery
description: a battery whose charger and discharger share one inverter
parameters:
  bus: null
  efficiency: 0.95
  max_hours: 8
  capital_cost_power: 0
  capital_cost_energy: 0
components:
  Bus:
    dc: {carrier: battery}
  Link:
    charger: {bus0: $bus, bus1: "@dc", efficiency: $efficiency, p_nom_extendable: true, capital_cost: $capital_cost_power}
    discharger: {bus0: "@dc", bus1: $bus, efficiency: $efficiency, p_nom_extendable: true}
  Store:
    store: {bus: "@dc", e_nom_extendable: true, e_cyclic: true, capital_cost: $capital_cost_energy}
math:
  expressions:
    battery_power:
      description: the net discharge of each instance
      dims: [scenario, snapshot, battery]
      expression: >-
        at(Link_p, by=battery_members[discharger])
        - at(Link_p, by=battery_members[charger])
  constraints:
    battery_inverter:
      description: the charger and the discharger share one rating
      dims: [scenario, battery]
      expression: >-
        at(Link_p_nom_effective, by=battery_members[charger])
        == at(Link_p_nom_effective, by=battery_members[discharger])
    battery_duration:
      description: the store holds at most `max_hours` of discharge
      dims: [scenario, battery]
      expression: >-
        at(Store_e_nom_effective, by=battery_members[store])
        <= battery_max_hours * at(Link_p_nom_effective, by=battery_members[discharger])
"""

In a template, `$bus` takes a parameter of the type, and `"@dc"` takes the name of another member of the same instance.
The math can use every entry of the spec. PyPSA adds three entries for the type (2.8 shows them):

- the dimension `battery`, over the instances;
- the relation `battery_members`, from each instance to its members. `at(Link_p, by=battery_members[charger])` reads the flow of the charger of each instance;
- the parameter `battery_max_hours`, because the math reads `max_hours`.

Register the type once. Then add instances with `n.add`, as you add a component.
Here the toy network gets a battery at each bus in place of its store:

In [15]:
def with_battery() -> pypsa.Network:
    n = toy()
    n.remove("Store", "storage")
    n.composites.register(BATTERY)
    costs = {"capital_cost_power": 20, "capital_cost_energy": 10}
    buses = ["north", "south"]
    n.add("battery", buses, bus=buses, max_hours=[1, 4], **costs)
    return n


n = with_battery()
print(n.composites, "|", n.composites["battery"])
n.c.links.static[["bus0", "bus1", "composite", "composite_member"]]

Composites ['battery'] | Composite 'battery' with 2 instances


,bus0,bus1,composite,composite_member
name,,,,
north charger,north,north dc,north,charger
south charger,south,south dc,south,charger
north discharger,north dc,north,north,discharger
south discharger,south dc,south,south,discharger


An instance is its members and nothing more. Each member is a plain component that names its instance in the column `composite`.

The build adds the math of the type. The north battery can store one hour of its rating, the south battery four hours:

In [16]:
n.optimize(include_expressions=True, log_to_console=False)
battery = n.composites["battery"]
members = battery.members
sizes = {
    "charger": n.links.p_nom_opt[members.charger].to_numpy(),
    "discharger": n.links.p_nom_opt[members.discharger].to_numpy(),
    "store": n.stores.e_nom_opt[members.store].to_numpy(),
    "max_hours": battery.static.max_hours,
}
pd.DataFrame(sizes, index=members.index).round(1)

,charger,discharger,store,max_hours
battery,,,,
north,39.9,39.9,39.9,1.0
south,21.1,21.1,21.1,4.0


In both instances the charger and the discharger have the same rating (`battery_inverter`).
The north store is at its limit of one hour (`battery_duration`).

`include_expressions=True` adds the named expressions of the spec to `n.model.expressions`.
So you can read `battery_power`, the net discharge of each battery, after the solve.
Statistics need nothing new: `groupby="composite"` groups the members by instance.

In [17]:
power = n.model.expressions["battery_power"].solution.sel(scenario="base", drop=True)
display(power.to_pandas().T.round(1))
n.statistics.capex(groupby="composite").round(1)

battery,north,south
snapshot,,
2030-01-01 00:00:00,-29.9,-12.2
2030-01-01 01:00:00,5.9,0.0
2030-01-01 02:00:00,34.0,21.1
2030-01-01 03:00:00,-12.1,-10.0


component  composite
Generator               8298.9
Link       north         797.6
           south         421.1
Store      north         398.8
           south         210.5
dtype: float64

A type is stored on the network as a stored override is (2.5).
The network protects the instances of a type. It refuses to remove a type that has instances. `n.remove` removes an instance with all its members:

In [18]:
show_error(lambda: n.composites.unregister("battery"))
n.remove("battery", "north")
print(n.composites["battery"])
print("links:", n.links.index.tolist())

ValueError: The battery instances ['north', 'south'] exist; remove them first with `n.remove('battery', ...)`.
Composite 'battery' with 1 instances
links: ['south charger', 'south discharger']


A composite type only adds entries, each with the name of the type as prefix. To change or remove an entry of the spec, use an override.

## 1.7 The math refuses input that it cannot handle

Some math holds only for some input. A `marginal_cost` curve, for example, needs a fixed capacity.
The spec states such a rule as an **assumption**, in the same file as the math that it protects:

In [19]:
show(s.to_dict()["assumptions"], "Generator_marginal_cost_curve_on_a_fixed_build")

Generator_marginal_cost_curve_on_a_fixed_build:
  holds: NOT Generator_p_nom_extendable AND Generator_p_nom > 0 AND Generator_p_nom < inf
  where: Generator_marginal_cost_curve
  description: a `marginal_cost` curve needs a fixed, positive, finite nominal capacity



`n.add` refuses a curve on an extendable unit. But the attribute can change later.
The assumption finds this before the build. The error quotes the assumption and names the unit:

In [20]:
n = toy()
peaker = {"carrier": "gas", "p_nom": 50, "marginal_cost": {0: 60, 0.5: 75, 1: 100}}
n.add("Generator", "peaker", bus="south", **peaker)
n.c.generators.static.loc["peaker", "p_nom_extendable"] = True
show_error(lambda: n.optimize(log_to_console=False))

SpecDataError: assumption 'Generator_marginal_cost_curve_on_a_fixed_build' does not hold for the data attached to 'Generator_p_nom', 'Generator_p_nom_extendable': a `marginal_cost` curve needs a fixed, positive, finite nominal capacity
  Not so at Generator='peaker', scenario='base'


---

# Part 2: How it works inside

This part uses internal names. They are not public API and can change.

## 2.1 The pipeline and the modules

```
YAML files ──_render.py──► default spec ──layers.py──► spec of the build ──┐
                                                                           ├─ linopy ─► model ─► naming.py, scaling.py
pypsa.Network ──checks.py──► sources.py and the data modules ──────────────┘
```

`n.optimize()` calls `build_model` in `build.py`. All other modules serve that one call.

| Module in `pypsa/optimization/spec/` | Role | Section |
|---|---|---|
| `_render.py` | Reads the YAML files and joins them into one spec. | 2.2, 2.3 |
| `layers.py` | Lays the patches over the spec in their order. Holds the caches. | 2.4 |
| `sources.py` | `NetworkSources`: the data that linopy asks for. | 2.6 |
| `_capacity.py`, `_commitment.py`, `_global_constraints.py`, `_maintenance.py`, `_multiport.py`, `_oneport.py`, `_passive_network.py`, `_piecewise.py`, `_ramping.py`, `_storage.py` | Data preparation, one module per topic. | 2.6, 2.7 |
| `build.py` | `build_model`: checks, spec, sources, linopy, layout, scaling. | 2.9 |
| `naming.py`, `scaling.py` | The layout and the scaling of the built model. | 2.3, 2.9 |
| `checks.py` | Checks on the network before the build. | 2.10 |
| `_docs.py` | Makes the formulation pages of the docs. | 2.12 |

| Module outside `spec/` | Role | Section |
|---|---|---|
| `pypsa/_registry.py` | Keeps composite types and stored overrides on the network. | 2.5 |
| `pypsa/optimization/overrides.py` | `n.optimize.overrides`, with the check of stored overrides. | 2.5 |
| `pypsa/composites.py` | `n.composites`, the patch and the data of a type. | 2.8 |
| `pypsa/optimization/_removed.py` | The error messages for removed names (1.2). | |

The size of the code and of the math:

In [21]:
def lines(path: Path) -> int:
    return len(path.read_text().splitlines())


yaml_files = sorted(SPEC_DIR.rglob("*.yaml"))
print(f"{len(yaml_files)} YAML files, {sum(map(lines, yaml_files)):,} lines")
sizes = {path.name: lines(path) for path in SPEC_DIR.glob("*.py")}
python = pd.Series(sizes, name="lines")
print(f"{len(python)} Python files, {python.sum():,} lines")
python.nlargest(6)

27 YAML files, 3,295 lines
19 Python files, 4,209 lines


sources.py                813
_docs.py                  558
_passive_network.py       404
layers.py                 346
_global_constraints.py    321
_render.py                278
Name: lines, dtype: int64

## 2.2 Rendering: traits and placeholders

**Problem.** Many components share math. Generator, Link and Store, for example, all have a capacity limit. One copy per component would drift.

**Solution.** The shared math is written once, in a **trait** file under `traits/`.
Each component class lists its traits in the class attribute `_formulation`:

In [22]:
from pypsa.optimization.spec._render import formulated, symbols

components = formulated(pypsa.Network())
traits = {c.name: dict.fromkeys(c._formulation, "x") for c in components}
pd.DataFrame(traits).fillna("")

,Generator,Load,Link,Store,StorageUnit,Line,Process,Transformer
capacity,x,,x,x,x,x,x,x
marginal_cost,x,,x,x,x,,x,
dispatch,x,,x,x,,,x,
oneport,x,x,,x,x,,,
rating,x,,x,,,,x,
commitment,x,,x,,,,x,
ramping,x,,x,,,,x,
maintenance,x,,x,,,,x,
operational_limit,x,,,x,x,,,
transmission_expansion,,,x,,,x,,


A trait uses **placeholders** in upper case. `_render.py` fills them in per component with `mathspec.rename`:

In [23]:
pd.DataFrame({c.name: symbols(c) for c in components}).fillna("").T

,COMPONENT,UNIT,NOM,POWER,COEF,CYCLIC,LEVEL
Generator,Generator,Generator,p_nom,p,,,
Load,Load,Load,p_nom,,,,
Link,Link,Link,p_nom,p,efficiency,,
Store,Store,Store,e_nom,p,,e_cyclic,e
StorageUnit,StorageUnit,StorageUnit,p_nom,p_dispatch,,cyclic_state_of_charge,state_of_charge
Line,Line,Line,s_nom,,,,
Process,Process,Process,p_nom,p,rate,,
Transformer,Transformer,Transformer,s_nom,,,,


A component has no entry for a placeholder that it has no value for. A Load, for example, has no `POWER`.
A trait that uses such a placeholder fails to render for that component. So a wrong trait list fails at once.

One constraint in `traits/rating.yaml`, and the keys that it becomes in the spec:

In [24]:
show(read("traits/rating.yaml")["constraints"], "COMPONENT_ext_p_upper")
print([key for key in s.constraints if key.endswith("_ext_p_upper")])

COMPONENT_ext_p_upper:
  name: COMPONENT-ext-p-upper
  description: an extendable unit that is not committable runs at most at its maximum of the chosen build
  dims:
  - scenario
  - snapshot
  - UNIT
  where: COMPONENT_ext AND NOT COMPONENT_committable
  expression: COMPONENT_p <= COMPONENT_p_max_pu * (COMPONENT_NOM_ext - COMPONENT_maintenance_pu * COMPONENT_maintenance_capacity)

['Generator_ext_p_upper', 'Link_ext_p_upper', 'Process_ext_p_upper', 'Store_ext_p_upper']


`render()` reads `system.yaml` first. Then, per component, it reads each trait file and the file of the component under `components/`.
Two files that declare the same entry raise an error.
Under `given:`, a trait lists the entries that it reads and that another file declares.

How much do traits save? The cell counts the entries that the files state, and the entries of the spec:

In [25]:
from pypsa.optimization.spec import written


def stated(path: Path) -> int:
    doc = yaml.safe_load(path.read_text())
    doc.pop("given", None)
    return len(written(doc))


def total(pattern: str) -> int:
    return sum(map(stated, SPEC_DIR.glob(pattern)))


patterns = ("system.yaml", "traits/*.yaml", "components/*.yaml")
counts = {pattern: total(pattern) for pattern in patterns}
counts |= {"all files": sum(counts.values()), "the spec": len(written(s.to_dict()))}
pd.Series(counts, name="entries")

system.yaml            83
traits/*.yaml         315
components/*.yaml      79
all files             477
the spec             1009
Name: entries, dtype: int64

The files state 477 entries, and the spec holds 1009. A correction in a trait reaches every component that uses the trait.

## 2.3 The fields `name:` and `scaling:`

**Problem.** Two properties of an entry are a matter of PyPSA, not of the math: its name in `n.model` and its numerical scaling.
`mathspec` knows neither, and it refuses a field that it does not know.

**Solution.** A YAML entry states both next to its math. `_render.py` takes the two fields out before `mathspec` reads the entry, and keeps them in two tables.
The entry in the file has the fields. The entry in the spec does not:

In [26]:
show(read("traits/capacity.yaml")["variables"], "COMPONENT_capital_cost_piecewise")
show(s.to_dict()["variables"], "Generator_capital_cost_piecewise")

COMPONENT_capital_cost_piecewise:
  name: COMPONENT-capital_cost_piecewise
  scaling: objective
  description: the build cost of an asset with a `capital_cost` curve
  dims:
  - UNIT
  where: COMPONENT_capital_cost_curve

Generator_capital_cost_piecewise:
  dims:
  - Generator
  where: Generator_capital_cost_curve
  description: the build cost of an asset with a `capital_cost` curve



`spec_names()` and `spec_scalings()` return the two tables. The layout step and the scaling step use them after the build (2.9):

In [27]:
from pypsa.optimization.spec import spec_names, spec_scalings

names = spec_names()
print("Generator_ext_p_upper ->", names["constraints"]["Generator_ext_p_upper"])
print("Generator_p_nom_ext   ->", names["variables"]["Generator_p_nom_ext"])
rules = pd.Series(spec_scalings()).rename_axis(["section", "key"])
rules.groupby("section").value_counts().unstack(fill_value=0)

Generator_ext_p_upper -> Generator-ext-p-upper
Generator_p_nom_ext   -> Generator-p_nom


,carrier,cycle,global,none,objective
section,,,,,
constraints,1,1,3,57,2
piecewise,0,0,0,0,24
variables,0,0,0,7,15


A scaling rule says where the factor of a column or a row comes from:

| Rule | Factor |
|---|---|
| `bus` | `Bus.scaling` of the bus of each unit |
| `objective` | the argument `objective_scaling` |
| `global` | `GlobalConstraint.scaling` |
| `cycle`, `carrier` | derived from the factors of the branches in a cycle, and of the assets of a carrier |
| `none` | 1 |

Most entries state no rule. They take one by their kind: `bus` for a continuous variable or a constraint over buses or over the units of a component on a bus, else `none` (`_column_rule` and `_rule` in `scaling.py`).

**Limit.** An override cannot set the two fields. An entry that an override adds keeps its key as its name and takes the rule `none`.
So with `Bus.scaling = 10`, the rows of the default constraints get the factor 10, and the rows of the added constraint keep the factor 1:

In [28]:
patch = {"constraints": {"Generator_ext_p_upper": {"scaling": "none"}}}
show_error(lambda: toy().optimize.spec(overrides=[patch]))

n = toy()
n.c.buses.static["scaling"] = 10.0
n.c.generators.static["full_load_hours_min"] = [2.2, 0.0]
m = n.optimize.create_model(overrides=[FULL_LOAD_HOURS])
rows = ("Bus-nodal_balance", "Generator-ext-p-upper", "Generator_full_load_hours")
factors = {name: float(m.constraints[name].scaling.max()) for name in rows}
pd.Series(factors, name="row factor")

LanguageError: constraints.Generator_ext_p_upper: unknown key 'scaling' in a constraint entry. Valid keys: description, dims, expression, where.


Bus-nodal_balance            10.0
Generator-ext-p-upper        10.0
Generator_full_load_hours     1.0
Name: row factor, dtype: float64

## 2.4 Layers and caches

**Problem.** A build can change the spec in five ways (the table in 1.5). The result must not depend on the code path, and a build must not render the YAML files again.

**Solution.** Every change is a patch, and `layers.py` lays the patches in one fixed order.

- *Linearised unit commitment* is not an `if` branch in the code. It is a folder of YAML patches, `linearized_uc/`, that `_render.py` lays over the files.
- *Piecewise options* become a patch that `_piecewise.py` writes.
- *Composite types, stored overrides and per-call overrides* are patches by nature.

Each layer has a label, as `the stored override 'typo'` in the error in 1.5. So an error names the layer that causes it.

The status variable shows what the patch folder does. The integer variable becomes a continuous share:

In [29]:
from pypsa.optimization.spec import spec

for linearized in (False, True):
    status = spec(linearized_unit_commitment=linearized).variables["Generator_status"]
    print(f"linearized_unit_commitment={linearized}")
    print(f"    domain {status.domain}, upper bound {status.bounds.upper}")
    print(f"    {status.description}")

linearized_unit_commitment=False
    domain integer, upper bound None
    how many modules of a committable unit are on; one module where it is not modular


linearized_unit_commitment=True
    domain continuous, upper bound 1.0
    the share of a committable unit that is on


`pypsa.optimization.spec.spec()` is the spec without a network: it has layers 1, 2 and 5 only.

**Caches.** To render the files and to lay patches takes time, so `layers.py` keeps the results:

| Cache | Holds | Key | Dropped |
|---|---|---|---|
| `_rendered`, `_base` | the spec from the files | `linearized_unit_commitment` | never in one process |
| `_PATCHED` | a spec with patches | the content of all patches, as JSON | the oldest, when there are more than `KEEP = 8` |

The key is the content, not the network. So two networks with the same patches share one spec object, and a dict and its YAML text give the same object:

In [30]:
first, second = toy(), toy()
first.optimize.overrides.register(HALF, name="half")
second.optimize.overrides.register(HALF, name="another_name")

as_dict, as_text = spec(overrides=[HALF]), spec(overrides=[yaml.safe_dump(HALF)])

print("same call twice:          ", spec() is spec())
print("dict and YAML text:       ", as_dict is as_text)
print("two networks, same patch: ", first.optimize.spec() is second.optimize.spec())
print("with and without a patch: ", as_dict is spec())
print("a spec is frozen:         ", spec().model_config["frozen"])

same call twice:           True
dict and YAML text:        True
two networks, same patch:  True
with and without a patch:  False
a spec is frozen:          True


A shared object is safe because a spec is frozen.

**What can surprise you:** `_rendered` never reads the files again. After you edit a YAML file of the spec, start the kernel again.

## 2.5 The registry and the fingerprints

**Problem.** Composite types and stored overrides change the model of a network. So they must go where the network goes: into a copy, a slice, a merged network and a file.
`n.meta` is the wrong place, because users own it and replace it.

**Solution.** The network keeps both in a private attribute, `n._registries`, by kind and by name (`pypsa/_registry.py`).
Its functions `carry`, `unite`, `saved` and `restore` take the content to a copy, a slice or a clustered network, to a merged network, and to a file and back.
A file holds it in the meta attribute under the key `"_pypsa"`, apart from the user's meta:

In [31]:
from pypsa.optimization.spec import fingerprint

n = toy()
n.meta = {"author": "a reviewer"}
n.optimize.overrides.register(HALF, name="half")
saved = n.export_to_netcdf()
meta = json.loads(saved.attrs["meta"])
record = meta["_pypsa"]["overrides"]["half"]
print("n.meta:    ", n.meta)
print("saved meta:", list(meta))
print("record:    ", {key: value for key, value in record.items() if key != "patch"})
print("now:       ", fingerprint("constraints.Generator_ext_p_upper"))

n.meta:     {'author': 'a reviewer'}
saved meta: ['author', '_pypsa']
record:     {'edits': {'constraints.Generator_ext_p_upper': '4f6221d25effb6f3'}, 'creates': []}
now:        4f6221d25effb6f3


**Problem.** A stored override is written against one PyPSA version. A later version can change the entry that the override edits. Then the same patch gives another model, and nobody sees it.

**Solution.** `register` records the entries that the patch edits, each with a **fingerprint**, and the entries that it creates.
A fingerprint is a hash of the entry in the default spec, without its description.
Each build compares the record with the spec. It refuses the override if an edited entry is gone or has another fingerprint, or if a created entry exists already.

The cell fakes a file from another PyPSA version: it changes the fingerprint in the saved meta. The build fails. The author then checks the override and registers it again with `overwrite=True`:

In [32]:
record["edits"]["constraints.Generator_ext_p_upper"] = "0" * 16
saved.attrs["meta"] = json.dumps(meta)
old = pypsa.Network()
old.import_from_netcdf(saved)
show_error(lambda: old.optimize(log_to_console=False))

old.optimize.overrides.register(HALF, name="half", overwrite=True)
old.optimize(log_to_console=False)
print(f"objective: {old.objective:,.2f}")

ValueError: The stored override 'half' changes ['constraints.Generator_ext_p_upper'], which PyPSA now defines differently. The formulation or the composite types changed since you registered it. Check the override and register it again with `overwrite=True`.


objective: 23,966.67


Limits of the check:

- It covers the entries that an override edits or creates. It does not cover the entries that the override only reads, as a variable in its expression.
- The fingerprint is of the default spec. A change in the description alone is no change. A rewrite by linearised unit commitment is no change (`test_an_entry_linearised_unit_commitment_rewrites_is_no_change`).

## 2.6 Sources: where the numbers come from

**Problem.** The spec says what a parameter is. It does not say where its values come from.

**Solution.** `NetworkSources` is a mapping. Its keys are the dimensions, relations and parameters that the spec declares.
It computes a value when linopy asks for it. `derivation(key)` finds the value in this order:

1. A function that is registered for the key with `@source(...)`.
2. For a parameter `<Component>_<attribute>`: the attribute of the component. Where the component has no such attribute, the parameter takes the value that its `missing:` field states.
3. For a relation `<Component>_<column>`: the static column.
4. For a dimension: the labels that the relations map into it.

The cell builds the sources of the toy network as `build_model` does, and counts the keys by their origin:

In [33]:
from pypsa.optimization.spec import NetworkSources
from pypsa.optimization.window import SnapshotWindow


def sources(n: pypsa.Network, multi_invest: bool = False) -> NetworkSources:
    n.calculate_dependent_values()
    window = SnapshotWindow.build(n, n.snapshots)
    index = (window.network_index, window.model_index)
    return NetworkSources(n, n.optimize.spec(), *index, multi_invest=multi_invest)


def origin(key: str) -> str:
    function = src.derivation(key).func
    if inspect.isfunction(function):
        return function.__module__.rsplit(".", 1)[-1] + ".py"
    return f"NetworkSources.{function.__name__}"


src = sources(toy())
origins = pd.Series({key: origin(key) for key in src})
by_name = (origins == "NetworkSources.attribute").sum()
registered = len(src) - by_name
print(f"{len(src)} keys: {by_name} by name, {registered} by a registered function")
origins.value_counts().sort_index().rename("keys")

374 keys: 119 by name, 255 by a registered function


NetworkSources.attribute    119
_capacity.py                 57
_commitment.py               27
_global_constraints.py       24
_maintenance.py              12
_multiport.py                10
_passive_network.py          28
_piecewise.py                31
_ramping.py                   3
_storage.py                  11
sources.py                   52
Name: keys, dtype: int64

An attribute that the math reads as it is needs no Python: 119 of the 374 keys. The other 255 keys have a registered function, because their data is derived: activity, weights, cycles, curves.

A registered function looks like this:

```python
@source("COMPONENT_capital_weight")
def _capital_weight(s: NetworkSources, name: str, c: Components) -> xr.DataArray: ...
```

A key with placeholders stands for one key per component, and the function gets the component as `c`.
`NetworkSources` keeps only the functions for keys that the spec declares. Section 2.7 follows this function.

A key that has no source fails with an error that says what to do. Here the override of 1.4 runs without its column:

In [34]:
show_error(lambda: toy().optimize(overrides=[FULL_LOAD_HOURS], log_to_console=False))

NotImplementedError: PyPSA has no data for the parameter 'Generator_full_load_hours_min': Generator has no attribute 'full_load_hours_min'. Add it as a column of the Generator components.


## 2.7 One key from start to end: `capital_weight`

`Generator_capital_weight` answers one question: **how often does the capital cost of a unit count in the objective?**

| Step | Where | What happens |
|---|---|---|
| 1. Declare | `traits/capacity.yaml` | The parameter `COMPONENT_capital_weight` is declared with placeholders. |
| 2. Render | `_render.py` | One copy per component with the trait `capacity`. |
| 3. Derive | `_capacity.py` | `@source("COMPONENT_capital_weight")` computes the values. |
| 4. Build | linopy | The entries that use the parameter get these values. |

Steps 1 to 3:

In [35]:
from pypsa.optimization.spec import _capacity

show(read("traits/capacity.yaml")["parameters"], "COMPONENT_capital_weight")
print([key for key in src if key.endswith("_capital_weight")], end="\n\n")
print(inspect.getsource(_capacity._capital_weight))

COMPONENT_capital_weight:
  description: the summed weights of the periods the asset is active in
  dims:
  - UNIT

['Generator_capital_weight', 'Line_capital_weight', 'Link_capital_weight', 'Process_capital_weight', 'StorageUnit_capital_weight', 'Store_capital_weight', 'Transformer_capital_weight']

@source("COMPONENT_capital_weight")
def _capital_weight(s: NetworkSources, name: str, c: Components) -> xr.DataArray:
    by_period = s.active_by_period(c)
    if not s.multi_invest:
        return by_period.any("period").astype(float)
    return (by_period * s["period_weighting_objective"]).sum("period")



Load has no trait `capacity`, so there is no `Load_capital_weight`.

With one period, the weight is 1 for an active unit. Investment periods make it visible.
Here three periods weigh 1.0, 0.7 and 0.5 in the objective, and three generators are active in different periods:

In [36]:
def periods() -> pypsa.Network:
    n = pypsa.Network()
    n.set_snapshots(pd.MultiIndex.from_product([[2030, 2040, 2050], range(2)]))
    n.set_investment_periods([2030, 2040, 2050])
    n.investment_period_weightings["objective"] = [1.0, 0.7, 0.5]
    n.add("Carrier", ["wind", "gas"])
    n.add("Bus", "home")
    n.add("Load", "demand", bus="home", p_set=50)
    names = ["wind-2030", "wind-2040", "gas"]
    data = {
        "carrier": ["wind", "wind", "gas"],
        "build_year": [2030, 2040, 2030],
        "lifetime": [15, 30, 50],
        "capital_cost": [80, 60, 40],
        "marginal_cost": [0, 0, 60],
    }
    n.add("Generator", names, bus="home", p_nom_extendable=True, **data)
    return n


n = periods()
src = sources(n, multi_invest=True)
weights = src.active_by_period(n.components.generators).to_pandas().T
weights["capital_weight"] = src["Generator_capital_weight"].to_pandas()

n.optimize(multi_investment_periods=True, log_to_console=False)
coefficients = n.model.objective.flat.groupby("vars").coeffs.sum()
labels = n.model.variables["Generator-p_nom"].labels.to_pandas()
weights["capital_cost x weight"] = n.generators.capital_cost * weights["capital_weight"]
weights["coefficient in the model"] = labels.map(coefficients)
weights

period,2030,2040,2050,capital_weight,capital_cost x weight,coefficient in the model
Generator,,,,,,
wind-2030,True,True,False,1.7,136.0,136.0
wind-2040,False,True,True,1.2,72.0,72.0
gas,True,True,True,2.2,88.0,88.0


`wind-2030` is active in 2030 and 2040, so its weight is 1.0 + 0.7 = 1.7.
The last two columns are equal: the objective of the built model uses the capital cost times this weight.
So the YAML entry, the Python function and the model agree.

## 2.8 Composites inside

**Problem.** A composite type brings math and data that the files of the spec do not have.

**Solution.** `pypsa/composites.py` makes two things per type:

- a **patch** (layer 3): the math of the definition, and the three entries that PyPSA adds (1.6);
- **sources** for these three entries, which `build_model` gives to `NetworkSources`.

A parameter of the math, as `max_hours`, is stored in a column of one member of each instance:

In [37]:
n = with_battery()
patch = pypsa.composites.patches(n)["battery"]
print({kind: list(entries) for kind, entries in patch.items()})
print("sources:", list(pypsa.composites.providers(n)))
dc_buses = n.composites["battery"].members.dc
n.c.buses.static.loc[dc_buses, ["carrier", "battery_max_hours"]]

{'dimensions': ['battery'], 'relations': ['battery_members'], 'parameters': ['battery_max_hours'], 'expressions': ['battery_power'], 'constraints': ['battery_inverter', 'battery_duration']}
sources: ['battery', 'battery_members', 'battery_max_hours']


,carrier,battery_max_hours
name,,
north dc,battery,1.0
south dc,battery,4.0


## 2.9 After linopy: the layout of the model

**Problem.** The spec names things as the math needs them: `Generator_p` over the dimension `Generator`. User code expects the names of PyPSA: `Generator-p` over `name`.

**Solution.** `build_model` gives linopy the PyPSA names (`rename=` of `Model.from_spec`, from the `name:` fields of 2.3). Then it runs three steps on the model:

| Step | What it does |
|---|---|
| `relabel` in `naming.py` | Names the dimension of the units `name`. Puts back the snapshot labels of the network. Removes the variables and constraints without a row. |
| `scale` in `scaling.py` | Sets the factors of columns and rows (2.3). It does nothing when all factors are 1. |
| `split_by_label` in `naming.py` | Splits the global constraints into one constraint per label. |

The cell adds a fixed generator and a global constraint to the toy network:

In [38]:
n = toy()
n.add("Generator", "old gas", bus="south", carrier="gas", p_nom=5, marginal_cost=90)
limit = {"carrier_attribute": "co2_emissions", "sense": "<=", "constant": 55}
n.add("GlobalConstraint", "co2_limit", **limit)
m = n.optimize.create_model()
print("spec: ", n.optimize.spec().variables["Generator_p"].dims)
print("model:", m.variables["Generator-p"].dims)
print([name for name in m.constraints if name.startswith("GlobalConstraint")])
m.variables["Generator-p_nom"]

spec:  ('Generator', 'scenario', 'snapshot')
model: ('name', 'scenario', 'snapshot')
['GlobalConstraint-co2_limit']


Variable (name: 3) - 1 masked entries
-------------------------------------
[wind]: Generator-p_nom[wind] ∈ [-inf, inf]
[gas]: Generator-p_nom[gas] ∈ [-inf, inf]
[old gas]: None

- The spec has one constraint per sense for all global constraints, as `GlobalConstraint_upper`. The model has `GlobalConstraint-co2_limit`, as before.
- `Generator-p_nom` lists all three generators and masks `old gas`, which is not extendable. Before, the variable left such units out. The variable runs over the whole dimension `Generator`, and its `where:` masks the units without a column.

## 2.10 Checks before the build

Checks on the input have two homes:

| Where | What |
|---|---|
| `assumptions:` in the YAML (1.7) | The input that the math cannot handle. |
| `check()` in `checks.py` | Warnings and errors on the data of the network that are not about the math. |

`check()` runs first in `build_model`:

In [39]:
from pypsa.optimization.spec import checks

print(inspect.getsource(checks.check))

def check(
    n: Network,
    sns: pd.Index,
    *,
    multi_invest: bool = False,
    linearized_unit_commitment: bool = False,
    piecewise_options: Sequence[PiecewiseOptions] = (),
) -> None:
    """Warn and raise on the network's data, before the build."""
    _check_invariants(n)
    _check_deprecations(n)
    _check_modules(n)
    if linearized_unit_commitment:
        _check_tightening(n)
    _check_cyclic(n, multi_invest)
    _check_energy_limits(n)
    _check_piecewise(n, piecewise_options)
    _check_objective(n, sns)



## 2.11 The guards

**Problem.** The PR replaces the model builder. A reviewer cannot check every entry by eye.

**Solution.** Three tests compare each build with recorded values in `test/data/equivalence/`:

| Test | Recorded | It fails when | Record again |
|---|---|---|---|
| `test_objective_matches_master` in `test_optimization_spec_equivalence.py` | `objectives.json`: the optimum of each network, solved on `master` | an optimum differs by more than `1e-6` (relative) | on a `master` checkout: `python test/test_optimization_spec_equivalence.py <commit>` |
| `test_spec_keys_match_lock` in `test_optimization_spec_lock.py` | `spec_keys.json`: each key of the spec with a hash of its entry | a key is new, gone, or has other content | `python test/test_optimization_spec_lock.py` |
| `test_model_digest_matches_lock` in the same file | `model_digests.json`: counts and sums of each built model, per variable and constraint (rows, nonzeros, bounds, right-hand sides, factors) | a count or a sum differs | the same command |

The first test compares with the old code. The other two find a change of the spec or of a model that nobody intended. The lock of the keys also backs the promise of stable keys (1.4).
The cell reads the three files (it needs a source checkout):

In [40]:
DATA = Path(pypsa.__file__).parents[1] / "test" / "data" / "equivalence"


def recorded(name: str) -> dict[str, Any]:
    return json.loads((DATA / name).read_text())


objectives = recorded("objectives.json")
cases = len(objectives["objectives"])
print(f"objectives.json: {cases} networks, solved on master at {objectives['commit']}")
for variant, kinds in recorded("spec_keys.json").items():
    print(f"spec_keys.json: {sum(map(len, kinds.values()))} keys for {variant}")
print(f"model_digests.json: {len(recorded('model_digests.json'))} models")

objectives.json: 24 networks, solved on master at a399eb4
spec_keys.json: 1009 keys for default
spec_keys.json: 42 keys for linearized_unit_commitment
model_digests.json: 22 models


`spec_keys.json` holds all 1009 keys of the default spec. For linearised unit commitment it holds only the 42 keys that differ from the default.

## 2.12 The generated docs

The formulation pages of the docs come from the same YAML files: one page per trait and per component.
`_docs.pages()` makes them, and `docs/hooks/spec_pages.py` calls it when the docs are built:

In [41]:
from pypsa.optimization.spec import _docs

pages = _docs.pages()
print(len(pages), "pages, as", list(pages)[:3], end="\n\n")
print(pages["traits/rating.md"].split("## Constraints")[0])

22 pages, as ['index.md', 'components/generator.md', 'components/load.md']

# rating

The trait `rating` states one formulation, in `traits/rating.yaml`, for each component that declares it. This page shows it for [`Generator`](../components/generator.md). Each component gives the trait these names:

| Component | Unit dimension | Nominal capacity | Power variable | Port coefficient |
| --- | --- | --- | --- | --- |
| `Generator` | `Generator` | `p_nom` | `p` |  |
| `Link` | `Link` | `p_nom` | `p` | `efficiency` |
| `Process` | `Process` | `p_nom` | `p` | `rate` |




## Where to look, and what is open

| To learn | Read |
|---|---|
| what the model is | the YAML files, or `n.optimize.spec()` (1.3) |
| which patches a build lays | `_stack` and `effective` in `layers.py` (2.4) |
| where a number comes from | the `@source(...)` call for its key (2.6) |
| why `n.model` looks as it does | `naming.py` and `scaling.py` (2.9) |
| what protects the change | the three tests in 2.11 |

Limits and open points, in one place:

- The PR needs linopy and mathspec versions that are not released (1.1).
- The layout of `n.model` changes. User code that reads or extends the model can break (1.2).
- Every key can change until the first release with overrides. After that, only the stable keys are fixed (1.4).
- An override cannot set a name or a scaling rule for the entries that it adds (2.3).
- The check of stored overrides covers what an override edits or creates, not what it reads (2.5).
- Python still holds logic: the data modules derive what the math reads (2.6), and two parts of the model are still built in Python (1.1).
- This notebook does not measure the build time against `master`.